# ResNet50 Model Development - Phase 1

Clean model development notebook:
- ResNet50 with ImageNet weights (frozen backbone)
- Custom classification head
- Configured for Phase 1 training (feature extraction only)

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)
import json
from pathlib import Path

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

TensorFlow version: 2.15.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Configuration

In [2]:
# Paths
PROJECT_ROOT = Path('/Users/nadaashraf/Desktop/FER copy')
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
LOGS_DIR = PROJECT_ROOT / 'logs'

# Create directories
MODELS_DIR.mkdir(exist_ok=True)
CHECKPOINTS_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

# Model parameters
IMG_SIZE = 224
NUM_CLASSES = 8
DROPOUT_RATE = 0.5

# Load emotion labels
with open(PROJECT_ROOT / 'class_indices.json', 'r') as f:
    class_indices = json.load(f)

EMOTIONS = list(class_indices.keys())
print(f'Emotions: {EMOTIONS}')

Emotions: ['Angry', 'Contempt', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


## Build ResNet50 Model

**Phase 1 Architecture:**
- ResNet50 backbone with ImageNet weights (FROZEN)
- Global Average Pooling
- Dense layer (256 units) with BatchNorm and Dropout
-  Output layer (8 classes, softmax)

In [3]:
def build_model_phase1(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """
    Build ResNet50 model for Phase 1 training (feature extraction).
    
    Phase 1:
    - Freeze ResNet50 backbone
    - Train only custom head
    - Establish baseline performance
    """
    # Load ResNet50 with ImageNet weights, exclude top
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze all ResNet50 layers (Phase 1)
    base_model.trainable = False
    
    # Build custom classification head
    x = base_model.output
    x = GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = Dense(256, activation='relu', name='fc1')(x)
    x = BatchNormalization(name='bn1')(x)
    x = Dropout(DROPOUT_RATE, name='dropout1')(x)
    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    # Create model
    model = Model(inputs=base_model.input, outputs=outputs, name='ResNet50_FER_Phase1')
    
    return model, base_model

# Build the model
model, base_model = build_model_phase1()

print(f'\nModel: {model.name}')
print(f'Total layers: {len(model.layers)}')
print(f'ResNet50 backbone frozen: {not base_model.trainable}')

2025-12-13 18:12:00.872979: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-12-13 18:12:00.873038: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2025-12-13 18:12:00.873054: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2025-12-13 18:12:00.873115: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-13 18:12:00.873161: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



Model: ResNet50_FER_Phase1
Total layers: 180
ResNet50 backbone frozen: True


In [4]:
# Count trainable parameters
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = sum([tf.size(w).numpy() for w in model.non_trainable_weights])
total_params = trainable_params + non_trainable_params

print(f'\nParameter Summary:')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)')
print(f'Non-trainable parameters: {non_trainable_params:,} ({non_trainable_params/total_params*100:.1f}%)')


Parameter Summary:
Total parameters: 24,115,336
Trainable parameters: 527,112 (2.2%)
Non-trainable parameters: 23,588,224 (97.8%)


## Model Summary

In [5]:
# Display model summary
model.summary()

Model: "ResNet50_FER_Phase1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 230, 230, 3)          0         ['input_1[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 112, 112, 64)         9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1_conv[0][0]']          
 on)                                                                            

## Compile Model

**Phase 1 Configuration:**
- Optimizer: Adam with learning rate 1e-3
- Loss: Categorical crossentropy
- Metrics: Accuracy

In [6]:
# Compile model for Phase 1
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Model compiled for Phase 1 training')
print(f'Optimizer: Adam (lr=1e-3)')
print(f'Loss: categorical_crossentropy')
print(f'Metrics: accuracy')

Model compiled for Phase 1 training
Optimizer: Adam (lr=1e-3)
Loss: categorical_crossentropy
Metrics: accuracy


## Prepare Callbacks

In [7]:
from datetime import datetime

# Create experiment ID
experiment_id = f"phase1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f'Experiment ID: {experiment_id}')

# Define callbacks
callbacks = [
    # Save best model
    ModelCheckpoint(
        filepath=str(CHECKPOINTS_DIR / f'{experiment_id}_best.keras'),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Early stopping
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # TensorBoard logging
    TensorBoard(
        log_dir=str(LOGS_DIR / experiment_id),
        histogram_freq=1,
        write_graph=True
    )
]

print(f'\nCallbacks configured:')
print('- ModelCheckpoint (best val_accuracy)')
print('- EarlyStopping (patience=10)')
print('- ReduceLROnPlateau (factor=0.5, patience=5)')
print('- TensorBoard logging')

Experiment ID: phase1_20251213_181229

Callbacks configured:
- ModelCheckpoint (best val_accuracy)
- EarlyStopping (patience=10)
- ReduceLROnPlateau (factor=0.5, patience=5)
- TensorBoard logging


## Save Model Architecture

In [8]:
# Save model architecture
model_config = {
    'experiment_id': experiment_id,
    'phase': 'Phase 1 - Feature Extraction',
    'architecture': 'ResNet50',
    'input_shape': [IMG_SIZE, IMG_SIZE, 3],
    'num_classes': NUM_CLASSES,
    'emotions': EMOTIONS,
    'backbone_frozen': True,
    'total_params': int(total_params),
    'trainable_params': int(trainable_params),
    'non_trainable_params': int(non_trainable_params),
    'dropout_rate': DROPOUT_RATE,
    'optimizer': 'Adam',
    'learning_rate': 1e-3,
    'loss': 'categorical_crossentropy'
}

with open(OUTPUT_DIR / f'{experiment_id}_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print(f'Model configuration saved to {OUTPUT_DIR / f"{experiment_id}_config.json"}')

Model configuration saved to /Users/nadaashraf/Desktop/FER copy/outputs/phase1_20251213_181229_config.json


## Summary

**Model Development Complete:**
- ✅ ResNet50 backbone loaded with ImageNet weights
- ✅ Backbone frozen for Phase 1 (feature extraction)
- ✅ Custom classification head added
- ✅ Model compiled with Adam optimizer
- ✅ Callbacks configured (checkpoint, early stopping, LR reduction)
- ✅ Architecture saved

**Phase 1 Training Info:**
- Trainable parameters: ~131K (only custom head)
- Non-trainable: ~23.5M (frozen ResNet50)
- Expected baseline: 50-65% validation accuracy

**Ready for:** Phase 1 training (30 epochs)